# VoxCPM2 Text-to-Speech with OpenVINO™

[VoxCPM2](https://github.com/OpenBMB/VoxCPM) is a 2-billion-parameter **tokenizer-free** diffusion autoregressive TTS model from OpenBMB / ModelBest.
It supports multilingual speech synthesis, voice cloning, and voice design through natural language style descriptions.

<img width="1025" height="625" alt="image" src="https://github.com/user-attachments/assets/7529bdef-659f-48c0-98a8-51353b9c98e8" />

The model pipeline includes:
- **AudioVAE** — encodes audio to a continuous latent space and decodes back to 48 kHz waveform
- **Local Encoder** — encodes latent audio patches to feature embeddings
- **Base LM** (28-layer MiniCPM4) — autoregressive language model that generates semantic features
- **Residual LM** (8-layer MiniCPM4) — refines the base LM output
- **LocDiT** — local diffusion transformer that converts LM features to audio latent patches via flow matching

In this tutorial we convert every learned component to OpenVINO IR and build a **pure-OpenVINO** inference pipeline.

#### Table of contents:
- [Prerequisites](#Prerequisites)
- [Download VoxCPM2 Weights](#Download-VoxCPM2-Weights)
- [Convert Model to OpenVINO](#Convert-Model-to-OpenVINO)
- [Create Inference Pipeline](#Create-Inference-Pipeline)
  - [Text-to-Speech](#Text-to-Speech)
  - [Voice Design](#Voice-Design)
  - [Controllable Voice Cloning](#Controllable-Voice-Cloning)
  - [Ultimate Cloning](#Ultimate-Cloning)
  - [Streaming](#Streaming)
- [Interactive Demo](#Interactive-Demo)


⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO. It may be fully supported and validated in the future.

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).


<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/voxcpm2-tts/voxcpm2-tts.ipynb" />


## Prerequisites
[back to top ⬆️](#Table-of-contents:)

Install dependencies and download model source code:

In [ ]:
import requests
from pathlib import Path

# Fetch utility helpers
for fname in ("notebook_utils.py", "cmd_helper.py", "pip_helper.py"):
    if not Path(fname).exists():
        r = requests.get(
            url=f"https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/{fname}",
        )
        open(fname, "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("voxcpm2-tts.ipynb")

In [ ]:
from cmd_helper import clone_repo
from pip_helper import pip_install

pip_install(
    "-q",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
    "torch>=2.4",
    "openvino>=2026.0",
    "nncf",
    "tokenizers",
    "transformers",
    "librosa",
    "soundfile",
    "scipy",
    "gradio>=4.19",
    "pydantic",
    "einops",
    "safetensors",
    "tqdm",
)

# Clone VoxCPM source code (needed for PyTorch model loading during conversion)
voxcpm_src_dir = clone_repo(
    "https://github.com/OpenBMB/VoxCPM.git",
    revision="79c0cf68ddb80895880dedbe7c1152d69670a451",
)

## Download VoxCPM2 Weights
[back to top ⬆️](#Table-of-contents:)

Download the VoxCPM2 model weights from Hugging Face. The model includes:
- `config.json` — model configuration
- `model.safetensors` — LM weights (~4 GB)
- `audiovae.safetensors` — AudioVAE weights (~200 MB)
- Tokenizer files

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

model_path = Path("VoxCPM2")
if not model_path.exists():
    snapshot_download(
        repo_id="openbmb/VoxCPM2",
        local_dir=str(model_path),
    )
    print(f"✅ VoxCPM2 weights downloaded to {model_path}")
else:
    print(f"✅ VoxCPM2 weights already exist at {model_path}")

## Convert Model to OpenVINO
[back to top ⬆️](#Table-of-contents:)

The conversion function loads the original PyTorch model and converts
all sub-models to OpenVINO IR format. The two autoregressive LMs
(base_lm, residual_lm) are made **stateful** so the KV-cache lives
inside the model and is managed automatically.

The flow-matching diffusion loop (**CFM Euler solver**) stays in Python
because it contains a dynamic `for` loop; only the inner DiT estimator
is an OV model.

Several small models are **merged** to reduce the total model count:
- `base_lm` now includes the FSQ layer (2 outputs: raw + quantized)
- `residual_lm` now includes `fusion_concat_proj` (takes `[B,T,2H]` input)
- `decode_heads` merges DiT projections + stop predictor into one model

Converted models (8 `.xml` files):

| # | Model | Notes |
|---|-------|-------|
| 1 | `embed_tokens` | Text token embedding |
| 2 | `feat_encoder` | Local encoder + projection (merged) |
| 3 | `base_lm` | 28-layer MiniCPM + FSQ, stateful KV-cache |
| 4 | `residual_lm` | Fusion proj + 8-layer MiniCPM, stateful KV-cache |
| 5 | `decode_heads` | DiT projections + stop predictor (merged) |
| 6 | `dit_estimator` | 12-layer DiT (non-causal) |
| 7 | `audio_vae_encoder` | Audio → latent |
| 8 | `audio_vae_decoder` | Latent → 48 kHz audio |

In [ ]:
import sys
from pathlib import Path

# from nncf import CompressWeightsMode

# Add VoxCPM source to path (needed for conversion)
voxcpm_src = Path("VoxCPM/src")
if str(voxcpm_src.resolve()) not in sys.path:
    sys.path.insert(0, str(voxcpm_src.resolve()))

from voxcpm2_tts_helper import convert_voxcpm2_model

model_path = Path("VoxCPM2")
ov_model_dir = Path("voxcpm2-ov")

convert_voxcpm2_model(
    model_path=str(model_path),
    output_dir=str(ov_model_dir),
    quantization_config=None,  # Set to {"mode": CompressWeightsMode.INT8_SYM} for INT8 weight compression on LM models
)

## Create Inference Pipeline
[back to top ⬆️](#Table-of-contents:)

### Select Inference Device

In [ ]:
from notebook_utils import device_widget

device = device_widget(default="CPU", exclude=["NPU"])
device

### Load OpenVINO Model
[back to top ⬆️](#Table-of-contents:)

In [ ]:
from voxcpm2_tts_helper import OVVoxCPM2Model

ov_model = OVVoxCPM2Model(
    model_dir=str(ov_model_dir),
    device=device.value,
)
print(f"Loaded OVVoxCPM2Model on {device.value}")
print(f"Output sample rate: {ov_model.sample_rate} Hz")

### Text-to-Speech
[back to top ⬆️](#Table-of-contents:)

Basic zero-shot synthesis — the model infers an appropriate voice from text content alone:

In [ ]:
import time
import IPython.display as ipd

test_text = "VoxCPM2 is a creative multilingual TTS model designed to generate highly realistic speech."

start = time.time()
wav, sr = ov_model.generate(text=test_text, cfg_value=2.0, inference_timesteps=10)
elapsed = time.time() - start

print(f"Generated {len(wav)/sr:.2f}s audio in {elapsed:.2f}s  (RTF={elapsed/(len(wav)/sr):.3f})")
ipd.Audio(wav, rate=sr)

### Voice Design
[back to top ⬆️](#Table-of-contents:)

Put a voice description in parentheses at the start of `text`, followed by the content to synthesize.
The model generates a matching voice **without** any reference audio:

In [ ]:
start = time.time()
wav, sr = ov_model.generate(
    text="(A young woman, gentle and sweet voice)Hello, welcome to VoxCPM2!",
    cfg_value=2.0,
    inference_timesteps=10,
)
elapsed = time.time() - start
print(f"Voice design: {len(wav)/sr:.2f}s audio in {elapsed:.2f}s  (RTF={elapsed/(len(wav)/sr):.3f})")
ipd.Audio(wav, rate=sr)

### Controllable Voice Cloning
[back to top ⬆️](#Table-of-contents:)

Clone a voice from a short reference audio clip.
Optionally add a style description in parentheses to steer emotion, pace, and expression while preserving the speaker's timbre:

In [ ]:
ref_wav = "VoxCPM/examples/reference_speaker.wav"

# Basic cloning
start = time.time()
wav, sr = ov_model.generate(
    text="This is a cloned voice generated by VoxCPM2 with OpenVINO.",
    reference_wav_path=ref_wav,
)
elapsed = time.time() - start
print(f"Controllable cloning: {len(wav)/sr:.2f}s audio in {elapsed:.2f}s  (RTF={elapsed/(len(wav)/sr):.3f})")
ipd.Audio(wav, rate=sr)

### Ultimate Cloning
[back to top ⬆️](#Table-of-contents:)

Provide both the reference audio **and** its exact transcript for maximum voice fidelity.
Pass the same clip to both `reference_wav_path` and `prompt_wav_path` for highest similarity.

> **Important:** `prompt_text` must be the **actual transcript** of the reference audio.
> A mismatched transcript will cause poor quality output.
> The official VoxCPM demo uses ASR to auto-transcribe; here we generate a reference
> audio first so the transcript is known.

In [ ]:
import soundfile as sf

# Step 1: Generate a reference audio with known text using voice cloning
ref_text = "Hello, this is a reference audio for voice cloning demonstration."
ref_gen_wav, ref_sr = ov_model.generate(
    text=ref_text,
    reference_wav_path=ref_wav,
)
# Save as reference for ultimate cloning
ref_file = "ultimate_clone_ref.wav"
sf.write(ref_file, ref_gen_wav, ref_sr)
print(f"Reference audio saved ({len(ref_gen_wav)/ref_sr:.2f}s)")

# Step 2: Ultimate cloning — use the reference audio + its exact transcript
start = time.time()
wav, sr = ov_model.generate(
    text="This is an ultimate cloning demonstration using VoxCPM2 with OpenVINO.",
    prompt_wav_path=ref_file,
    prompt_text=ref_text,
    reference_wav_path=ref_file,
)
elapsed = time.time() - start
print(f"Ultimate cloning: {len(wav)/sr:.2f}s audio in {elapsed:.2f}s  (RTF={elapsed/(len(wav)/sr):.3f})")
ipd.Audio(wav, rate=sr)

### Streaming
[back to top ⬆️](#Table-of-contents:)

Generate speech in streaming mode — audio chunks are yielded as they're produced,
enabling low-latency playback. Each chunk contains ~80 ms of audio (3840 samples at 48 kHz):

In [ ]:
import numpy as np

chunks = []
start = time.time()
for chunk in ov_model.generate_streaming(text="Streaming is easy with VoxCPM2 on OpenVINO!"):
    chunks.append(chunk)
wav = np.concatenate(chunks)
elapsed = time.time() - start

print(f"Streaming: {len(chunks)} chunks, {len(wav)/sr:.2f}s audio in {elapsed:.2f}s  (RTF={elapsed/(len(wav)/sr):.3f})")
ipd.Audio(wav, rate=ov_model.sample_rate)

## Interactive Demo
[back to top ⬆️](#Table-of-contents:)

Launch a Gradio interface supporting three modes:
- **Voice Design** — control voice style via text description
- **Controllable Cloning** — clone from reference audio with style override
- **Ultimate Cloning** — transcript-guided audio continuation

In [ ]:
from gradio_helper import make_demo

demo = make_demo(ov_model)

try:
    demo.launch(debug=False)
except Exception:
    demo.launch(share=True, debug=False)

# If you are launching remotely, specify server_name and server_port:
# demo.launch(server_name='your server name', server_port='server port number')